In [ ]:
import torch
import torch.nn as nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [ ]:
mean_gray = 0.1307
std_gray = 0.3081

transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(mean_gray,), std=(std_gray,))
])

train_dataset = datasets.MNIST(root="./data", train=True, transform=transforms, download=True)
test_dataset = datasets.MNIST(root="./data", train=False, transform=transforms)

In [ ]:
batch_size = 100
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1, stride=1)
        self.batchnorm1 = nn.BatchNorm2d(num_features=8)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=2)
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=32, kernel_size=5, padding=2, stride=1)
        self.batchnorm2 = nn.BatchNorm2d(num_features=32)
        self.fc1 = nn.Linear(in_features=7 * 7 * 32, out_features=600)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(in_features=600, out_features=10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.batchnorm1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.conv2(x)
        x = self.batchnorm2(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = x.view(-1, 7 * 7 * 32)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [6]:
model = CNN()

CUDA = torch.cuda.is_available()
if CUDA:
    model = model.cuda()

loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr= 0.01)

epochs = 10

train_loss = []
test_loss = []
train_accuracy = []
test_accuracy = []

for epoch in range(epochs):
    iter_loss = 0
    correct = 0
    iterations = 0
    model.train()

    for i, (images, labels) in enumerate(train_loader):
        if CUDA:
            images = images.cuda()
            labels = labels.cuda()

        output = model(images)
        (_, predicted) = torch.max(output, 1)
        correct += (predicted == labels).sum()

        loss = loss_func(output, labels)
        iter_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        iterations += 1

    train_loss.append(iter_loss / iterations)
    train_accuracy.append((correct / len(train_dataset)) * 100)

    iter_loss = 0
    correct = 0
    iterations = 0
    model.eval()

    for i, (images, labels) in enumerate(test_loader):
        if CUDA:
            images = images.cuda()
            labels = labels.cuda()

        output = model(images)
        (_, predicted) = torch.max(output, 1)
        correct += (predicted == labels).sum()

        loss = loss_func(output, labels)
        iter_loss += loss.item()

        iterations += 1

    test_loss.append(iter_loss / iterations)
    test_accuracy.append((correct / len(test_dataset)) * 100)

    print(
        'Epoch {}/{}, Training Loss: {:.3f}, Training Accuracy: {:.3f}, Testing Loss: {:.3f}, Testing Acc: {:.3f}'.format(
            epoch + 1, epochs, train_loss[-1], train_accuracy[-1], test_loss[-1], test_accuracy[-1]))


Epoch 1/10, Training Loss: 0.489, Training Accuracy: 86.945, Testing Loss: 0.155, Testing Acc: 96.050
Epoch 2/10, Training Loss: 0.161, Training Accuracy: 95.417, Testing Loss: 0.097, Testing Acc: 97.170
Epoch 3/10, Training Loss: 0.114, Training Accuracy: 96.660, Testing Loss: 0.077, Testing Acc: 97.730
Epoch 4/10, Training Loss: 0.092, Training Accuracy: 97.313, Testing Loss: 0.065, Testing Acc: 97.970
Epoch 5/10, Training Loss: 0.079, Training Accuracy: 97.667, Testing Loss: 0.057, Testing Acc: 98.240
Epoch 6/10, Training Loss: 0.071, Training Accuracy: 97.860, Testing Loss: 0.051, Testing Acc: 98.470
Epoch 7/10, Training Loss: 0.064, Training Accuracy: 98.115, Testing Loss: 0.048, Testing Acc: 98.530
Epoch 8/10, Training Loss: 0.059, Training Accuracy: 98.222, Testing Loss: 0.048, Testing Acc: 98.530
Epoch 9/10, Training Loss: 0.055, Training Accuracy: 98.367, Testing Loss: 0.043, Testing Acc: 98.660
Epoch 10/10, Training Loss: 0.050, Training Accuracy: 98.518, Testing Loss: 0.042,

In [7]:
image = test_dataset[30][0].view(1, 1, 28, 28)
label = test_dataset[30][1]

model.eval()

if CUDA:
    image = image.cuda()
    model = model.cuda()

output = model(image)
(_, predicted) = torch.max(output, 1)
print("Predicted: {}".format(predicted.item()))
print("Actual: {}".format(label))

Predicted: 3
Actual: 3
